# Part I: Trajectory Optimization (50 pts)

### Linear dynamics (10 pts)

In this problem, we will use **quadratic programming** to solve a trajectory optimization problem for the planar quadrotor. We begin by defining the linear approximation of the dynamics we derived in class.

State: $\bar{x} = [x, y, \theta, \dot{x}, \dot{y}, \dot{\theta}]^T$. Input: $\bar{u} = [u_1, u_2]^T$, where $u_1 = F_1 + F_2$ and $u_2 = (F_2 - F_1)L$. [Note: we are transposing row vectors to get column vectors; hence $\bar{x}$ has dimension $6 \times 1$ and $\bar{u}$ has dimension $2 \times 1$.]

As discussed in lecture, the reference state we use for linearization is the hover state: $\bar{x}_0 = [0,0,0,0,0,0]^T, \ \bar{u}_0 = [mg, 0]^T$. We can then write the linear dynamics as: 
$$\dot{\bar{x}} = A\bar{x} + B(\bar{u} - \bar{u}_0) = A\bar{x} + B\bar{u} + \bar{c}, \qquad \bar{c} = -B\bar{u}_0 = [0, 0, 0, 0, -g, 0]^T.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, LinearConstraint, Bounds

# Physical parameters of planar quadrotor
m, I, L, g = 0.486, 0.00383, 0.25, 9.81   # kg, kg*m^2, m, m/s^2

In [ ]:
# TODO: Fill out the A and B matrices (from lecture)
A = np.array([[]])
B = np.array([[]])

### Discretization (10 pts)
Using Euler integration with time step $\Delta t$: $\;\bar{x}_{k+1} = \bar{x}_k + \Delta t\,(A\bar{x}_k + B \bar{u}_k + \bar{c}) = A_d \bar{x}_k + B_d \bar{u}_k + \bar{c}_d$.

In [ ]:
dt = 0.05          # time step [s]
N  = 40            # number of steps (horizon T = N*dt = 2 s)
nx, nu = 6, 2

# TODO: Euler discretization; fill in the variables below as numpy arrays
Ad =  
Bd = 
cd = 

### Solving QPs with `scipy.optimize.minimize`
`minimize` can solve quadratic programs of the general form

$$
\begin{aligned}
\min_{\bar{z}} \quad & \tfrac{1}{2} \bar{z}^T H \bar{z} + \bar{q}^T \bar{z} \\
\text{s.t.} \quad & \bar{l}_C \le C \bar{z} \le \bar{u}_C \qquad &\leftarrow \texttt{LinearConstraint(C, $\bar{l}_C$, $\bar{u}_C$)} \\
& \bar{l}_z \le \bar{z} \le \bar{u}_z \qquad &\leftarrow \texttt{Bounds($\bar{l}_z$, $\bar{u}_z$)}
\end{aligned}
$$

where $\bar{z} \in \mathbb{R}^{n_z}$ is the vector of decision variables (i.e., the variables we are trying to find). A few tricks make this form very flexible:
- **Equality constraints** $C\bar{z} = \bar{b}$: set both sides equal, `LinearConstraint(C, b, b)`.
- **One-sided or missing bounds**: use `-np.inf` / `np.inf` for any entry that is unbounded.
- The solver needs the cost and (for convenience) its derivatives, passed as functions of $\bar{z}$:

| argument | what to pass |
|---|---|
| `fun` | cost $\tfrac{1}{2} \bar{z}^T H \bar{z} + \bar{q}^T \bar{z}$ |
| `x0` | initial guess for $\bar{z}$ (zeros is fine for a QP) |
| `jac` | gradient $H \bar{z} + \bar{q}$ |
| `hess` | Hessian $H$ |
| `method` | `'trust-constr'` (supports linear constraints + bounds) |
| `constraints` | list of `LinearConstraint` objects |
| `bounds` | a `Bounds` object |

The solution is returned in `res.x`. Here is a toy example with two variables:
$\;\min\, z_1^2 + z_2^2 \;$ s.t. $\; z_1 + z_2 = 1,\;\; z_1 \ge 0.7$.

In [ ]:
H_toy = 2 * np.eye(2)                 # z1^2 + z2^2 = 0.5 z^T (2I) z
q_toy = np.zeros(2)
eq  = LinearConstraint(np.array([[1, 1]]), 1, 1)     # z1 + z2 = 1
bnd = Bounds([0.7, -np.inf], [np.inf, np.inf])      # z1 >= 0.7, z2 free

res_toy = minimize(lambda z: 0.5 * z @ H_toy @ z + q_toy @ z, x0=np.zeros(2),
                   jac=lambda z: H_toy @ z + q_toy, hess=lambda z: H_toy,
                   method='trust-constr', constraints=[eq], bounds=bnd)
print(res_toy.x)   # expect approx. [0.7, 0.3] (numerical solver, so not exact)

### Setting up the trajectory optimization QP (30 pts)

We want to find a sequence of states and control inputs with the following constraints:
- **Dynamics**: the trajectory must obey the linear dynamics of the drone.
- **Starting state**: the drone must start at the specified location. We will take this to be the origin.
- **Goal state**: the drone must end at the goal state. We will take this to be the state where the drone is hovering at x location = 1m. 
- **State bounds**: We want the (x,y) position of the drone to remain within some spcified limits; in particular x should remain within the range (-0.1, 1.1) and y in the range (-0.2, 0.2).

Our objective ("cost" function) is to minimize total control effort along the trajectory. Our problem can thus be stated as:

$$\min_{\bar{x}_{0:N},\, \bar{u}_{0:N-1}} \sum_{k=0}^{N-1} \|\bar{u}_k\|^2 \quad \text{s.t.} \quad \bar{x}_{k+1} = A_d\bar{x}_k + B_d \bar{u}_k + \bar{c}_d,\;\; \bar{x}_0 = \bar{x}_{start},\;\; \bar{x}_N = \bar{x}_{goal},\;\; x_{min} \le x_k \le x_{max},\;\; y_{min} \le y_k \le y_{max}.$$

To put it in the general form of the QP above, we stack all states and inputs into a single decision vector

$$\bar{z} = [\,\underbrace{\bar{x}_0,\, \bar{x}_1,\, \dots,\, \bar{x}_N}_{(N+1)\cdot 6},\; \underbrace{\bar{u}_0,\, \bar{u}_1,\, \dots,\, \bar{u}_{N-1}}_{N \cdot 2}\,] \in \mathbb{R}^{n_z}.$$

Then each piece maps onto the general form as follows:
- **Cost:** Defined by $\bar{q} = \bar{0}$, and $H$ (which you will figure out).
- **Equality constraints** should capture the linear dynamics of the drone, the start location, and the goal location.

- **Bounds:** $\bar{l}_z, \bar{u}_z$ are $\pm\infty$ everywhere except the entries of $\bar{z}$ corresponding to $x_k$ and $y_k$.

Below, the helpers `ix(k)` and `iu(k)` below return the slice of $\bar{z}$ holding $\bar{x}_k$ and $\bar{u}_k$, so e.g. `C[rows, ix(k)] = -Ad` places $-A_d$ in the right columns.

In [ ]:
x_start = np.zeros(nx)                          # hover at origin
x_goal  = np.array([1.0, 0, 0, 0, 0, 0])        # hover 1 m to the right
x_lim, y_lim = (-0.1, 1.1), (-0.2, 0.2)         # position bounds [m]

nz = (N + 1) * nx + N * nu # Total number of decision variables
ix = lambda k: slice(k * nx, (k + 1) * nx)                              # indices of x_k in z
iu = lambda k: slice((N + 1) * nx + k * nu, (N + 1) * nx + (k + 1) * nu)  # indices of u_k in z

In [ ]:
# TODO: Cost  0.5 z^T H z + q^T z  (only inputs are penalized)
q = np.zeros(nz)
H = 

# TODO: Equality constraints  C z = b  (N dynamics block-rows + initial state constraint + final state constraint)
# Hint: The "slice" function may be useful here again
C = # numpy array
b = # numpy array

# TODO: Bounds  l_z <= z <= u_z  (only x and y are bounded)
l_z, u_z = # numpy array

In [ ]:
# Do not modify
# Solve the QP
res = minimize(lambda z: 0.5 * z @ H @ z + q @ z, x0=np.zeros(nz),
               jac=lambda z: H @ z + q, hess=lambda z: H,
               method='trust-constr',
               constraints=[LinearConstraint(C, b, b)],
               bounds=Bounds(l_z, u_z))
print(res.message, '| cost =', res.fun)

X = res.x[:(N + 1) * nx].reshape(N + 1, nx)
U = res.x[(N + 1) * nx:].reshape(N, nu)
t = np.arange(N + 1) * dt

In [ ]:
# Do not modify
# Plot optimized trajectory

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Path in the x-y plane with drone snapshots
ax[0].plot(X[:, 0], X[:, 1], 'k--', lw=1)
for k in range(0, N + 1, 8):
    x, y, th = X[k, :3]
    ax[0].plot([x - L * np.cos(th), x + L * np.cos(th)],
               [y - L * np.sin(th), y + L * np.sin(th)], 'b-', lw=2, alpha=0.4 + 0.6 * k / N)
ax[0].set(xlabel='x [m]', ylabel='y [m]', title='Trajectory', aspect='equal', ylim=(-0.4, 0.4))

# States
ax[1].plot(t, X[:, 0], label='x [m]')
ax[1].plot(t, X[:, 1], label='y [m]')
ax[1].plot(t, X[:, 2], label=r'$\theta$ [rad]')
ax[1].set(xlabel='t [s]', title='States'); ax[1].legend()

# Inputs
ax[2].step(t[:-1], U[:, 0], 'C0', where='post')
ax[2].axhline(m * g, color='C0', ls=':', lw=1)   # hover thrust
ax[2].set(xlabel='t [s]', title='Inputs', ylabel=r'$u_1$ [N]', ylim=(0, 2 * m * g))
ax2 = ax[2].twinx()
ax2.step(t[:-1], U[:, 1], 'C1', where='post')
ax2.set_ylabel(r'$u_2$ [N m]', color='C1')

plt.tight_layout(); plt.show()

# Part II: Linearization (20 pts)

In lecture, we saw how to approximate nonlinear dynamics with a linear approximation. For more complicated systems (e.g., 3D quadrotor), it is too tedious to derive the linear dynamics by hand. Therefore, we will use a computer algebra system (CAS) to save us a lot of time in computing derivatives. Specifically, we will be using the [SymPy](https://docs.sympy.org/latest/index.html) package to help us along the way. If you have experience with Mathematica or MATLAB's Symbolic Toolkit, SymPy offers many of the same features but in a Python interface, and even allows us to convert the symbolic functions that we will derive into efficient numerical ones.

We will demonstrate how to use SymPy to symbolically linearize the planar quadrotor model. We also suggest looking through the [brief tutorial](https://docs.sympy.org/latest/tutorials/intro-tutorial/index.html#intro-tutorial) given in the SymPy documentation, which will cover most of what you will need for this assignment. Again, the dynamics of the planar quadrotor are:

$$\begin{align}\ddot{x} &= -\frac{u_1}{m}\sin\theta\\ \ddot{y} &= \frac{u_1}{m}\cos\theta - g\\ \ddot{\theta} &= \frac{u_2}{I}\end{align}$$

We begin by importing the functions we need from SymPy.

In [ ]:
import sympy as sp
import numpy as np
from sympy.physics.vector import dynamicsymbols as dynamicsymbols

Next, we define the symbolic variables we need to describe the equations of motion. The function ``dynamicsymbols`` creates symbols that vary in time, i.e. ``dynamicsymbols('x')`` will create a symbol $x(t)$ as opposed to $x$.

In [ ]:
m, g, I, r, t = sp.symbols('m g I r t')
u1, u2  = sp.symbols('u1 u2')
x, y, theta = dynamicsymbols('x y theta')

We also define some variables as shorthand for the time derivatives of our state variables.

In [ ]:
x_dot = sp.diff(x, t)
y_dot = sp.diff(y, t)
theta_dot = sp.diff(theta, t)

Now, we write out the equations of motion. Note that `sp.Matrix` is used to create both matrices and vectors in a manner similar to `np.array`.

In [ ]:
# TODO: Fill out the state, input, and dynamics

# Fill in using the variables x, y, theta, x_dot, y_dot, theta_dot
state = sp.Matrix([])
input = sp.Matrix([])

# Equations of motion of the drone written in first-order form
# Fill in using the variables x, y, theta, x_dot, y_dot, theta_dot
dynamics = sp.Matrix([]) # 6 x 1

Finally, we differentiate and plug in numerical values via the `subs` method. In this case, the hover state is chosen to be the $0$ vector and the hover input is $u_1 = m g$. We'll leave the parameters of the system as symbolic values for now so we can see the structure of the result.

In [ ]:
A = dynamics.jacobian(state)
B = dynamics.jacobian(input)

In [ ]:
A

In [ ]:
B

In [ ]:
# TODO
A.subs() # Use the subs function to plug in the reference state and control inputs

In [ ]:
# TODO
B.subs() # Use the subs function to plug in the reference state and control inputs

The last thing we need to discuss is turning the A and B matrices in the previous cells into `np.ndarray` types. The reason we want to do this is that after we replace the remaining system parameters with numerical values, SymPy will represent these values in a way that is very precise, but incredibly inefficient for the numerical work we need to do. Instead, we want to work with NumPy's numerical arrays, i.e. `np.ndarray`. These cannot handle symbolic values like SymPy's matrices and will not operate at same level of precision, but they are much more efficient for computational purposes.

In [ ]:
A_arr = np.array(A.subs([(u1, m * g), (theta, 0), (m, 0.03), (g, 9.81)])).astype(float)

B_arr = np.array(B.subs([(theta, 0), (m, 0.03), (I, 1.419e-5)])).astype(float)

In each of these two lines, we are doing the following. First, we substitute in all the hover state and parameter numerical values as before. Next, we call `np.array` to convert from `sp.Matrix` to `np.ndarray`. However, we still have to tell numpy which data type to use to represent the numerical values in this matrix. To do so, we call the method `np.ndarray.astype()` with the type we want, `float`. You can see these matrices have a different numerical representation now:

In [ ]:
print(A_arr)
print()
print(B_arr)

# Submission Instructions

Please submit your completed Lab3.ipynb file to Gradescope "HW3: Coding". 

Also, please don't forget to turn in your solution to the written portion of the assignment to Gradescope "HW3: Theory".